# 05 · Phase 4 — REST API, JSON-LD and Python client

**Input:** `webapp/db.sqlite3` (Phase 3 loaded) · **Code under test:** `catalog/api_views.py`, `serializers.py`, `filters.py`, `src/threatgraph/jsonld.py`, `api_client.py`
**Output:** `docs/phase4_validation_report.json`

The notebook starts the Django server **by itself** on a free port, talks to it **over real HTTP** with the client, and compares every answer with an **independent** source: the clean CSV files read with pandas (no ORM, no API code).

| Check family | Question |
|---|---|
| http | status codes, read-only (405), bad filters (400), content negotiation (406, `ld+json`) |
| paging | does pagination return every row exactly once? |
| filters | does each filter return exactly the set that pandas computes from the CSV files? |
| content | for all 1,391 CVEs and 799 techniques: is every field of the JSON identical to the CSV tables? |
| jsonld | no undefined terms, correct datatypes, missing ≠ empty, graph answers = REST answers = pandas answers |
| merge | do two separate JSON-LD documents about the same mapping merge into one graph? (interoperability) |
| dataset | DCAT/PROV description: hashes, versions and counts equal to `sources.yaml` and the Phase-2 manifest |
| perf | no N+1 queries (constant number of SQL queries per page/detail) |
| negative | do the checks FAIL when we feed them broken data? |

In [1]:
import sys, os, io, json, time, socket, subprocess, atexit, statistics, tempfile, copy
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "sources.yaml").exists())
WEBAPP = ROOT / "webapp"
sys.path[:0] = [str(ROOT / "src"), str(WEBAPP)]
os.environ["DJANGO_SETTINGS_MODULE"] = "threatgraph_site.settings"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"          # Jupyter event loop vs Django ORM (same workaround as notebook 04)
import django; django.setup()
import pandas as pd, requests
from IPython.display import display
from django.db import connection
from django.test import Client as DjangoClient
from django.test.utils import CaptureQueriesContext, setup_test_environment
from rdflib import Graph, Literal, Namespace, RDF, URIRef
from rdflib.namespace import XSD
from pyld import jsonld as pyld
from threatgraph import config, cleaning as C, jsonld as LD
from threatgraph.api_client import ThreatGraphClient, ApiError, JSON, JSONLD
from threatgraph.integrity import sha256_bytes
import catalog.models as M
pd.set_option("display.max_columns", 30, "display.max_colwidth", 70, "display.width", 200)
SRC = config.load_sources(); CLEAN = config.CLEAN_DIR
clean_manifest = json.loads((CLEAN / "manifest.json").read_text())
assert json.loads((ROOT / "docs" / "phase3_validation_report.json").read_text())["all_passed"], "Phase 3 gate not passed"
assert M.Vulnerability.objects.count() == 1391, "database is empty: run  python manage.py load_clean_data  in webapp/"
TG = Namespace(LD.VOCAB); DCT = Namespace("http://purl.org/dc/terms/")

checks = []
def check(family, name, ok, detail=""):
    checks.append({"family": family, "name": name, "ok": bool(ok), "detail": str(detail)[:300]}); return bool(ok)

# --- start the server (own port, no autoreload, log in a temp file) and make sure it is stopped at the end
def free_port():
    s = socket.socket(); s.bind(("127.0.0.1", 0)); p = s.getsockname()[1]; s.close(); return p
PORT = free_port(); BASE = f"http://127.0.0.1:{PORT}"
log = tempfile.NamedTemporaryFile("w+", suffix=".log", delete=False)
server = subprocess.Popen([sys.executable, "manage.py", "runserver", f"127.0.0.1:{PORT}", "--noreload"], cwd=WEBAPP,
                          stdout=log, stderr=subprocess.STDOUT)
atexit.register(lambda: server.poll() is None and server.terminate())
for _ in range(100):
    try:
        if requests.get(BASE + "/stats/", timeout=1).status_code == 200: break
    except requests.ConnectionError: time.sleep(0.3)
else:
    raise RuntimeError("server did not start:\n" + Path(log.name).read_text()[-1500:])
api = ThreatGraphClient(BASE)
from concurrent.futures import ThreadPoolExecutor
import threading
_tl = threading.local()
def pfetch(fn, keys, workers=8):
    # one client (= one HTTP session) per thread; returns {key: result}
    def one(k):
        if not hasattr(_tl, "c"): _tl.c = ThreatGraphClient(BASE)
        return k, fn(_tl.c, k)
    with ThreadPoolExecutor(workers) as ex: return dict(ex.map(one, keys))
print("server up at", BASE, "| Django", django.get_version())

server up at http://127.0.0.1:45381 | Django 5.2.18


## 1 · Independent reference: the clean CSV tables
Everything below is compared with these pandas frames. They come from `data/clean/` (hash-verified in Phase 2), **not** from the database or the API.

In [2]:
T = {t: C.read_clean(t, CLEAN) for t in clean_manifest["tables"]}
kev, nvd, kc, nc = T["kev_vulnerability"], T["nvd_cve"], T["kev_cwe"], T["nvd_cve_cwe"]
tech, tac, tt, pm, pr = T["attack_technique"], T["attack_tactic"], T["attack_technique_tactic"], T["pkev_mapping"], T["pkev_mapping_reference"]
CVES = sorted(kev.cve_id)
nz = lambda x: None if pd.isna(x) else x
day = lambda x: None if pd.isna(x) else str(pd.Timestamp(x).date())
ts = lambda x: None if pd.isna(x) else pd.Timestamp(x).tz_localize("UTC") if pd.Timestamp(x).tzinfo is None else pd.Timestamp(x).tz_convert("UTC")
print({t: len(df) for t, df in T.items()})
print(kev.dtypes.to_dict()); print(nvd[["published", "cvss_v31_score"]].dtypes.to_dict())

{'kev_vulnerability': 1391, 'kev_cwe': 1328, 'nvd_cve': 1391, 'nvd_cve_cwe': 2523, 'attack_tactic': 14, 'attack_technique': 799, 'attack_technique_tactic': 1051, 'pkev_mapping': 1183, 'pkev_mapping_reference': 2256}
{'cve_id': <StringDtype(storage='python', na_value=<NA>)>, 'vendor_project': <StringDtype(storage='python', na_value=<NA>)>, 'product': <StringDtype(storage='python', na_value=<NA>)>, 'vulnerability_name': <StringDtype(storage='python', na_value=<NA>)>, 'date_added': dtype('<M8[us]'), 'due_date': dtype('<M8[us]'), 'short_description': <StringDtype(storage='python', na_value=<NA>)>, 'required_action': <StringDtype(storage='python', na_value=<NA>)>, 'ransomware_use': <StringDtype(storage='python', na_value=<NA>)>, 'notes': <StringDtype(storage='python', na_value=<NA>)>, 'due_before_added': dtype('bool'), 'src_row': Int64Dtype()}
{'published': datetime64[us, UTC], 'cvss_v31_score': dtype('float64')}


## 2 · HTTP behaviour: status codes, read-only, bad input, content negotiation

In [3]:
rows = []
def probe(name, r, status, ctype=None):
    ok = r.status_code == status and (ctype is None or r.headers.get("Content-Type", "").startswith(ctype))
    rows.append({"request": name, "status": r.status_code, "content-type": r.headers.get("Content-Type", ""), "expected": status, "ok": ok}); return ok
for p in ["/api/", "/api/vulnerabilities/", "/api/techniques/", "/api/tactics/", "/api/cwes/", "/api/dataset/", "/api/context/"]:
    probe("GET " + p, api.request(p), 200)
probe("GET unknown CVE", api.request("/api/vulnerabilities/CVE-1999-0001/"), 404)
probe("GET unknown technique", api.request("/api/techniques/T9999/"), 404)
probe("GET malformed technique id", api.request("/api/techniques/banana/"), 404)
probe("GET page beyond the end", api.request("/api/vulnerabilities/", {"page": 999}), 404)
for m in ("POST", "PUT", "PATCH", "DELETE"):
    probe(f"{m} list (read-only API)", api.request("/api/vulnerabilities/", method=m), 405)
probe("DELETE detail", api.request("/api/vulnerabilities/CVE-2024-34102/", method="DELETE"), 405)
for q in [{"ransomware_use": "maybe"}, {"date_added_after": "yesterday"}, {"min_cvss": "abc"}, {"severity": "EXTREME"}, {"has_attack": "perhaps"}]:
    probe(f"GET bad filter {q}", api.request("/api/vulnerabilities/", q), 400)
probe("Accept: application/xml", api.request("/api/vulnerabilities/CVE-2024-34102/", accept="application/xml"), 406)
probe("Accept: application/json", api.request("/api/vulnerabilities/CVE-2024-34102/"), 200, "application/json")
probe("Accept: application/ld+json", api.request("/api/vulnerabilities/CVE-2024-34102/", accept=JSONLD), 200, "application/ld+json")
probe("?format=jsonld (browser-like Accept: */*)", api.request("/api/vulnerabilities/CVE-2024-34102/", {"format": "jsonld"}, accept="*/*"), 200, "application/ld+json")
probe("?format=jsonld but Accept: application/json -> contradiction", api.request("/api/vulnerabilities/CVE-2024-34102/", {"format": "jsonld"}), 406)
probe("?has_attack=1 / 0 accepted", api.request("/api/vulnerabilities/", {"has_attack": "1"}), 200)
probe("Accept: */* (default = JSON)", api.request("/api/vulnerabilities/CVE-2024-34102/", accept="*/*"), 200, "application/json")
http = pd.DataFrame(rows); display(http)
check("http", f"{int(http.ok.sum())}/{len(http)} status-code and content-negotiation probes behave as designed", http.ok.all(), http[~http.ok].to_dict("records"))
r = api.request("/api/vulnerabilities/CVE-2024-34102/", {"format": "jsonld"}, accept="*/*")
check("http", "JSON and JSON-LD of the same URL are both valid JSON with different media types", r.json() and r.headers["Content-Type"].startswith("application/ld+json"))

,request,status,content-type,expected,ok
0,GET /api/,200,application/json,200,True
1,GET /api/vulnerabilities/,200,application/json,200,True
2,GET /api/techniques/,200,application/json,200,True
3,GET /api/tactics/,200,application/json,200,True
4,GET /api/cwes/,200,application/json,200,True
5,GET /api/dataset/,200,application/json,200,True
6,GET /api/context/,200,application/json,200,True
7,GET unknown CVE,404,application/json,404,True
8,GET unknown technique,404,application/json,404,True
9,GET malformed technique id,404,text/html; charset=utf-8,404,True


True

## 3 · Pagination and counts

In [4]:
first = api.get("/api/vulnerabilities/")
pages = list(api.iter_pages("/api/vulnerabilities/", page_size=200))
ids = [v["cve_id"] for p in pages for v in p["results"]]
big = api.get("/api/vulnerabilities/", {"page_size": 100000})
check("paging", "count == 1,391 and default page size is 50", first["count"] == 1391 and len(first["results"]) == 50, (first["count"], len(first["results"])))
check("paging", "walking all pages gives every CVE exactly once, equal to the clean KEV table", len(ids) == len(set(ids)) == 1391 and set(ids) == set(kev.cve_id), (len(ids), len(set(ids))))
check("paging", "page_size is capped at its maximum (asked 100,000)", len(big["results"]) == 200 and big["next"], len(big["results"]))
check("paging", "first page has previous=None, last page has next=None", pages[0]["previous"] is None and pages[-1]["next"] is None and len(pages) == 7, len(pages))
tech_ids = [t["technique_id"] for t in api.iter_techniques()]
check("paging", "techniques: 799 distinct ids == attack_technique table", len(tech_ids) == len(set(tech_ids)) == 799 and set(tech_ids) == set(tech.technique_id))
check("paging", "tactics: 14 rows; CWEs: 176 rows", len(api.tactics()) == 14 and len(api.cwes()) == 176, (len(api.tactics()), len(api.cwes())))
check("paging", "no filter and ?search= of a CVE id: client.count() works without downloading everything", api.count() == 1391 and api.count(search="CVE-2024-34102") == 1)

True

## 4 · Filters against an independent recount
Each filter is asked over HTTP (all pages) and the **set of CVE ids** is compared with the set computed by pandas from the CSV files.
Two subtle points are tested on purpose:
* a filter that joins a one-to-many table (technique, tactic, CWE) must not return the same CVE twice;
* two filters on the **same** multi-valued relation (`technique=…` and `mapping_type=…`) are applied one after the other, so they mean *"has a mapping to T… AND has a mapping of type …"* (possibly two different mappings), not *"has one mapping with both"*.

In [5]:
sev = nvd.set_index("cve_id")
S = lambda df: set(df.cve_id)
pm_tac = pm.merge(tt, on="technique_id").merge(tac[["tactic_id", "shortname"]], on="tactic_id")
fil = lambda col, s: kev[kev[col].str.contains(s, case=False, regex=False)]
cases = [
  ("vendor_project=micro",              {"vendor_project": "micro"},               S(fil("vendor_project", "micro"))),
  ("product=exchange",                  {"product": "exchange"},                   S(fil("product", "exchange"))),
  ("ransomware_use=known",              {"ransomware_use": "known"},               S(kev[kev.ransomware_use == "known"])),
  ("date_added 2023-01-01..2023-12-31", {"date_added_after": "2023-01-01", "date_added_before": "2023-12-31"},
                                         S(kev[(kev.date_added >= "2023-01-01") & (kev.date_added <= "2023-12-31")])),
  ("cwe=CWE-79",                        {"cwe": "CWE-79"},                         S(kc[kc.cwe_id == "CWE-79"])),
  ("technique=T1190",                   {"technique": "T1190"},                    S(pm[pm.technique_id == "T1190"])),
  ("tactic=initial-access",             {"tactic": "initial-access"},              S(pm_tac[pm_tac.shortname == "initial-access"])),
  ("mapping_type=primary_impact",       {"mapping_type": "primary_impact"},        S(pm[pm.mapping_type == "primary_impact"])),
  ("severity=CRITICAL",                 {"severity": "CRITICAL"},                  set(nvd[nvd.cvss_v31_severity == "CRITICAL"].cve_id)),
  ("min_cvss=9",                        {"min_cvss": 9},                           set(nvd[nvd.cvss_v31_score >= 9].cve_id)),
  ("has_attack=true",                   {"has_attack": "true"},                    S(pm)),
  ("has_attack=false",                  {"has_attack": "false"},                   set(kev.cve_id) - S(pm)),
  ("technique=T1190 + severity=CRITICAL", {"technique": "T1190", "severity": "CRITICAL"},
                                         S(pm[pm.technique_id == "T1190"]) & set(nvd[nvd.cvss_v31_severity == "CRITICAL"].cve_id)),
  ("technique=T1190 + mapping_type=primary_impact (two joins!)", {"technique": "T1190", "mapping_type": "primary_impact"},
                                         S(pm[pm.technique_id == "T1190"]) & S(pm[pm.mapping_type == "primary_impact"])),
  ("search=fortinet",                   {"search": "fortinet"},
                                         set(kev[kev.cve_id.str.contains("fortinet", case=False) | kev.vendor_project.str.contains("fortinet", case=False)
                                                 | kev["product"].str.contains("fortinet", case=False) | kev.vulnerability_name.str.contains("fortinet", case=False)].cve_id)),
  ("nothing matches (vendor=zzzz)",     {"vendor_project": "zzzz"},                set()),
]
res = []
for name, q, expected in cases:
    got = [v["cve_id"] for v in api.iter_vulnerabilities(**q)]
    ok = len(got) == len(set(got)) and set(got) == expected and api.count(**q) == len(expected)
    res.append({"filter": name, "API rows": len(got), "pandas rows": len(expected), "duplicates": len(got) - len(set(got)), "ok": ok})
res = pd.DataFrame(res); display(res)
check("filters", f"{int(res.ok.sum())}/{len(res)} filters return exactly the pandas set (no duplicates, count() agrees)", res.ok.all(), res[~res.ok].to_dict("records"))
order = [v["date_added"] for v in api.iter_vulnerabilities(ordering="-date_added")]
check("filters", "?ordering=-date_added is non-increasing over all pages", order == sorted(order, reverse=True) and len(order) == 1391)
top = api.get("/api/vulnerabilities/", {"ordering": "-n_mappings", "page_size": 1})["results"][0]
check("filters", "?ordering=-n_mappings: first row has the maximum number of PKEV mappings", top["n_mappings"] == pm.cve_id.value_counts().max(), (top["cve_id"], top["n_mappings"]))
tt_ = list(api.iter_techniques(ordering="-n_vulnerabilities"))
exp_top = pm.groupby("technique_id").cve_id.nunique().max()
check("filters", "techniques: ?ordering=-n_vulnerabilities gives the technique with most CVEs; subtechnique filter and tactic filter work",
      tt_[0]["n_vulnerabilities"] == exp_top and api.count("techniques", is_subtechnique="true") == int(tech.is_subtechnique.sum())
      and api.count("techniques", tactic="initial-access") == len(tt.merge(tac[tac.shortname == "initial-access"], on="tactic_id")), (tt_[0]["technique_id"], exp_top))

,filter,API rows,pandas rows,duplicates,ok
0,vendor_project=micro,350,350,0,True
1,product=exchange,17,17,0,True
2,ransomware_use=known,291,291,0,True
3,date_added 2023-01-01..2023-12-31,187,187,0,True
4,cwe=CWE-79,27,27,0,True
5,technique=T1190,157,157,0,True
6,tactic=initial-access,260,260,0,True
7,mapping_type=primary_impact,359,359,0,True
8,severity=CRITICAL,473,473,0,True
9,min_cvss=9,473,473,0,True


True

## 5 · Content: every field of every record against the CSV tables
For **all 1,391 CVEs** (JSON detail) and **all 799 techniques** the comparison function checks KEV fields, NVD fields (incl. CVSS), CWE sets, PKEV mappings with their evidence and the technique/tactic data.
`compare_*` returns the list of differences; an empty list means identical.

In [6]:
kev_i, nvd_i = kev.set_index("cve_id"), nvd.set_index("cve_id")
kc_g = kc.groupby("cve_id").cwe_id.apply(set).to_dict()
nc_g = {k: g for k, g in nc.groupby("cve_id")}
pm_g = {k: g for k, g in pm.groupby("cve_id")}
pr_g = {k: set(zip(g.reference, g.is_http_url)) for k, g in pr.groupby("mapping_id")}
tech_i = tech.set_index("technique_id")
tact_of = tt.merge(tac, on="tactic_id").groupby("technique_id").apply(lambda g: set(zip(g.tactic_id, g.shortname)), include_groups=False).to_dict()
cwe_info = {c: c not in C.NONINFORMATIVE_CWE for c in set(kc.cwe_id) | set(nc.cwe_id)}

def compare_vulnerability(d, cve):
    diff = []
    k = kev_i.loc[cve]
    for col in ["vendor_project", "product", "vulnerability_name", "short_description", "required_action", "ransomware_use"]:
        if d[col] != k[col]: diff.append(col)
    if d["date_added"] != day(k.date_added) or d["due_date"] != day(k.due_date): diff.append("dates")
    if d["due_before_added"] != bool(k.due_before_added): diff.append("due_before_added")
    if (d["notes"] or None) != nz(k.notes): diff.append("notes")
    if {c["cwe_id"] for c in d["kev_cwes"]} != kc_g.get(cve, set()): diff.append("kev_cwes")
    if any(c["is_informative"] != cwe_info[c["cwe_id"]] for c in d["kev_cwes"]): diff.append("kev_cwes.is_informative")
    n, r = d["nvd"], nvd_i.loc[cve]
    if n is None: return diff + ["nvd missing"]
    if (n["status"], n["description_en"], n["n_references"]) != (r.vuln_status, r.description_en, int(r.n_references)): diff.append("nvd scalars")
    if ts(n["published"]) != ts(r.published) or ts(n["last_modified"]) != ts(r.last_modified): diff.append("nvd timestamps")
    if pd.isna(r.cvss_v31_score):
        if n["cvss_v31"] is not None: diff.append("cvss_v31 should be null")
    elif n["cvss_v31"] is None or (n["cvss_v31"]["score"], n["cvss_v31"]["severity"], n["cvss_v31"]["vector"], n["cvss_v31"]["source"], n["cvss_v31"]["type"]) != (
            r.cvss_v31_score, r.cvss_v31_severity, r.cvss_v31_vector, r.cvss_v31_source, r.cvss_v31_type): diff.append("cvss_v31")
    if n["cvss_v2_score"] != nz(r.cvss_v2_score): diff.append("cvss_v2")
    exp_nc = set() if cve not in nc_g else set(zip(nc_g[cve].cwe_id, nc_g[cve].source, nc_g[cve].type))
    if {(c["cwe_id"], c["source"], c["type"]) for c in n["cwes"]} != exp_nc: diff.append("nvd cwes")
    exp_m = pm_g.get(cve)
    got_m = {(m["mapping_id"], m["technique_id"], m["mapping_type"], m["capability_group"], m["comments"], m["status"]) for m in d["attack"]}
    want_m = set() if exp_m is None else {(int(a), b, c, e, nz(f), g) for a, b, c, e, f, g in zip(exp_m.mapping_id, exp_m.technique_id, exp_m.mapping_type, exp_m.capability_group, exp_m.comments, exp_m.status)}
    if got_m != want_m or len(d["attack"]) != len(want_m): diff.append("mappings")
    for m in d["attack"]:
        if {(x["reference"], x["is_http_url"]) for x in m["references"]} != pr_g.get(m["mapping_id"], set()): diff.append(f"evidence of mapping {m['mapping_id']}")
        if m["technique_name"] != tech_i.loc[m["technique_id"], "name"]: diff.append("technique_name")
        if {(x["tactic_id"], x["shortname"]) for x in m["tactics"]} != tact_of.get(m["technique_id"], set()): diff.append("tactics")
    return diff

t0 = time.time()
docs = pfetch(lambda c, k: c.vulnerability(k), CVES)
print(f"fetched {len(docs)} JSON details in {time.time()-t0:.1f}s")
bad = {cve: compare_vulnerability(d, cve) for cve, d in docs.items()}; bad = {k: v for k, v in bad.items() if v}
check("content", "1,391 CVE detail documents are identical to the clean tables (KEV, NVD, CWE, PKEV, evidence, tactics)", not bad and len(docs) == 1391, list(bad.items())[:3])
miss = [c for c, d in docs.items() if d["nvd"] and d["nvd"]["cvss_v31"] is None]
check("content", "missing != empty: CVEs without CVSS v3.1 show cvss_v31 = null (not 0, not empty)", len(miss) == int(nvd.cvss_v31_score.isna().sum()) == 2, miss)
unm = [c for c, d in docs.items() if not d["attack"]]
check("content", "972 CVEs have attack == [] (= not mapped by PKEV) and 419 have mappings", len(unm) == 972 and 1391 - len(unm) == 419, len(unm))

fetched 1391 JSON details in 16.7s


True

In [7]:
sub_of = tech[tech.parent_technique_id.notna()].groupby("parent_technique_id").technique_id.apply(set).to_dict()
pm_t = {k: set(zip(g.mapping_id.astype(int), g.cve_id, g.mapping_type)) for k, g in pm.groupby("technique_id")}
def compare_technique(d, tid):
    diff = []; r = tech_i.loc[tid]
    if (d["name"], d["status"], d["is_subtechnique"], d["stix_id"]) != (r["name"], r["status"], bool(r.is_subtechnique), r.stix_id): diff.append("scalars")
    if nz(d["description"]) != nz(r.description): diff.append("description")
    if ts(d["created"]) != ts(r.created) or ts(d["modified"]) != ts(r.modified): diff.append("timestamps")
    if (d["parent_technique"] or {}).get("technique_id") != nz(r.parent_technique_id): diff.append("parent")
    if {s["technique_id"] for s in d["subtechniques"]} != sub_of.get(tid, set()): diff.append("subtechniques")
    if {(x["tactic_id"], x["shortname"]) for x in d["tactics"]} != tact_of.get(tid, set()): diff.append("tactics")
    if {(m["mapping_id"], m["cve_id"], m["mapping_type"]) for m in d["mappings"]} != pm_t.get(tid, set()): diff.append("mappings")
    return diff
t0 = time.time()
tdocs = pfetch(lambda c, k: c.technique(k), sorted(tech.technique_id))
print(f"fetched {len(tdocs)} technique details in {time.time()-t0:.1f}s")
badt = {t: compare_technique(d, t) for t, d in tdocs.items()}; badt = {k: v for k, v in badt.items() if v}
check("content", "799 technique detail documents are identical to the clean tables (incl. parent, sub-techniques, tactics, CVEs)", not badt and len(tdocs) == 799, list(badt.items())[:3])
# the two views of the PKEV mapping must agree: CVE -> techniques  ==  technique -> CVEs
v_side = {(m["mapping_id"], cve, m["technique_id"]) for cve, d in docs.items() for m in d["attack"]}
t_side = {(m["mapping_id"], m["cve_id"], t) for t, d in tdocs.items() for m in d["mappings"]}
check("content", "the CVE view and the technique view describe the same 1,183 mappings", v_side == t_side and len(v_side) == 1183, (len(v_side), len(t_side)))

fetched 799 technique details in 7.0s


True

## 6 · JSON-LD
**What JSON-LD adds:** the same facts, but every key is tied (through the `@context`) to a globally unique IRI, and every thing has an `@id` — the public IRI of the CVE, of the technique, of the CWE.
Two documents from different places that talk about `https://attack.mitre.org/techniques/T1190/` are therefore talking about the **same node**, with no extra mapping work.

**Pitfall that is tested here:** a JSON-LD processor *silently drops* every key the context does not define. So a typo or a forgotten term would not raise an error; the data would just vanish.

In [8]:
t0 = time.time()
ld = pfetch(lambda c, k: c.vulnerability(k, jsonld=True), CVES)
print(f"fetched {len(ld)} JSON-LD documents in {time.time()-t0:.1f}s")
print(json.dumps({k: v for k, v in ld["CVE-2024-34102"].items() if k != "@context"}, indent=1)[:2300], "...")

fetched 1391 JSON-LD documents in 13.2s
{
 "@id": "https://nvd.nist.gov/vuln/detail/CVE-2024-34102",
 "@type": "Vulnerability",
 "cveId": "CVE-2024-34102",
 "apiUrl": "http://127.0.0.1:45381/api/vulnerabilities/CVE-2024-34102/",
 "vendorProject": "Adobe",
 "product": "Commerce and Magento Open Source",
 "title": "Adobe Commerce and Magento Open Source Improper Restriction of XML External Entity Reference (XXE) Vulnerability",
 "dateAdded": "2024-07-17",
 "dueDate": "2024-08-07",
 "shortDescription": "Adobe Commerce and Magento Open Source contain an improper restriction of XML external entity reference (XXE) vulnerability that allows for remote code execution.",
 "requiredAction": "Apply mitigations per vendor instructions or discontinue use of the product if mitigations are unavailable.",
 "ransomwareUse": "unknown",
 "notes": "https://helpx.adobe.com/security/products/magento/apsb24-40.html;  https://nvd.nist.gov/vuln/detail/CVE-2024-34102",
 "dueBeforeAdded": false,
 "weakness": [
 

In [9]:
undef = {c: LD.undefined_terms({k: v for k, v in d.items() if k != "@context"}) for c, d in ld.items()}; undef = {k: v for k, v in undef.items() if v}
check("jsonld", "all 1,391 JSON-LD documents use only terms defined in the @context (nothing would be dropped)", not undef, list(undef.items())[:2])
check("jsonld", "one context term per IRI (otherwise expand->compact could rename keys)", not LD.terms_sharing_an_iri(), LD.terms_sharing_an_iri())
check("jsonld", "every document carries the same @context as /api/context/ and as jsonld.CONTEXT", all(d["@context"] == LD.CONTEXT for d in ld.values()) and api.context() == LD.CONTEXT)
check("jsonld", "@id of every CVE is its NVD IRI; of every technique its attack.mitre.org IRI",
      all(d["@id"] == f"https://nvd.nist.gov/vuln/detail/{c}" for c, d in ld.items())
      and LD.technique_iri("T1059.001") == "https://attack.mitre.org/techniques/T1059/001/" and LD.technique_iri("T1190") == "https://attack.mitre.org/techniques/T1190/")

# one big graph: the 1,391 nodes in one document (the context is processed once)
big = {"@context": LD.CONTEXT, "@graph": [{k: v for k, v in d.items() if k != "@context"} for d in ld.values()]}
t0 = time.time(); G = Graph(); G.parse(data=json.dumps(big), format="json-ld"); print(f"rdflib parsed {len(G):,} triples in {time.time()-t0:.1f}s")
q = lambda s: [tuple(str(x) if x is not None else None for x in row) for row in G.query(s)]
PFX = "PREFIX tg: <%s> PREFIX dct: <http://purl.org/dc/terms/> PREFIX xsd: <http://www.w3.org/2001/XMLSchema#> " % LD.VOCAB
n_vuln = int(q(PFX + "SELECT (COUNT(DISTINCT ?v) AS ?n) WHERE { ?v a tg:Vulnerability }")[0][0])
n_map  = int(q(PFX + "SELECT (COUNT(DISTINCT ?m) AS ?n) WHERE { ?m a tg:Mapping }")[0][0])
n_tech = int(q(PFX + "SELECT (COUNT(DISTINCT ?t) AS ?n) WHERE { ?m tg:technique ?t }")[0][0])
check("jsonld", "graph: 1,391 vulnerabilities, 1,183 mapping nodes, 155 distinct techniques (same as pandas)", (n_vuln, n_map, n_tech) == (1391, 1183, 155), (n_vuln, n_map, n_tech))
by_type = dict(q(PFX + "SELECT ?t (COUNT(DISTINCT ?m) AS ?n) WHERE { ?m tg:mappingType ?t } GROUP BY ?t"))
check("jsonld", "graph: mappings per mapping_type == pandas value_counts", {k: int(v) for k, v in by_type.items()} == pm.mapping_type.value_counts().to_dict(), by_type)
bad_dt = int(q(PFX + "SELECT (COUNT(*) AS ?n) WHERE { ?v tg:dateAdded ?d FILTER(datatype(?d) != xsd:date) }")[0][0])
bad_sc = int(q(PFX + "SELECT (COUNT(*) AS ?n) WHERE { ?c tg:baseScore ?s FILTER(datatype(?s) != xsd:decimal) }")[0][0])
check("jsonld", "datatypes: all dateAdded are xsd:date, all CVSS scores xsd:decimal", bad_dt == 0 and bad_sc == 0)
empty = int(q(PFX + 'SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o FILTER(isLiteral(?o) && STR(?o) = "") }')[0][0])
n_score = int(q(PFX + "SELECT (COUNT(DISTINCT ?n) AS ?n) WHERE { ?n tg:cvssV31 ?c . ?c tg:baseScore ?s }")[0][0])
check("jsonld", "missing != empty: no empty-string literal anywhere; CVSS v3.1 node only for the 1,389 CVEs that have one",
      empty == 0 and n_score == int(nvd.cvss_v31_score.notna().sum()) == 1389, (empty, n_score))
# THREE-WAY: SPARQL on the graph == REST filter == pandas
sparql = q(PFX + "SELECT DISTINCT ?id WHERE { ?v dct:identifier ?id ; a tg:Vulnerability ; tg:hasMapping ?m . ?m tg:technique/tg:tactic/tg:shortname ?s "
                 "FILTER(?s = 'initial-access') ?v tg:nvdRecord/tg:cvssV31/tg:severity 'CRITICAL' }")
via_sparql = {r[0] for r in sparql}
via_rest = {v["cve_id"] for v in api.iter_vulnerabilities(tactic="initial-access", severity="CRITICAL")}
via_pd = S(pm_tac[pm_tac.shortname == "initial-access"]) & set(nvd[nvd.cvss_v31_severity == "CRITICAL"].cve_id)
check("jsonld", "three-way: CRITICAL CVEs mapped to an initial-access technique — SPARQL on JSON-LD == REST filter == pandas", via_sparql == via_rest == via_pd and len(via_pd) > 0, (len(via_sparql), len(via_rest), len(via_pd)))

rdflib parsed 66,970 triples in 1.4s


True

In [10]:
# the expand -> compact round trip must give the document back: proves that no key is lost and the context is consistent
sample = ["CVE-2024-34102", *[c for c in CVES if docs[c]["nvd"]["cvss_v31"] is None], *unm[:3], *CVES[::60]]
def strip(x):
    return x
lost = []
for c in dict.fromkeys(sample):
    doc = ld[c]
    back = pyld.compact(pyld.expand(doc), LD.CONTEXT)
    a = json.dumps({k: v for k, v in doc.items() if k != "@context"}, sort_keys=True)
    b = json.dumps({k: v for k, v in back.items() if k != "@context"}, sort_keys=True)
    if a != b: lost.append(c)
check("jsonld", f"PyLD expand->compact round trip returns the document unchanged ({len(set(sample))} CVEs incl. the 2 without CVSS)", not lost, lost[:3])
# each document alone is also a valid, self-contained graph
solo = [len(Graph().parse(data=json.dumps(ld[c]), format="json-ld")) for c in sample]
check("jsonld", "each sampled document parses alone with rdflib (self-contained @context) and gives > 10 triples", min(solo) > 10, (min(solo), max(solo)))
# JSON-LD says the same as the JSON: facts read back from the graph
mism = []
for c in sample:
    g1 = Graph().parse(data=json.dumps(ld[c]), format="json-ld"); v = URIRef(LD.vulnerability_iri(c)); d = docs[c]
    if set(map(str, g1.objects(v, DCT.identifier))) != {c}: mism.append((c, "id"))
    maps = {str(m) for m in g1.objects(v, TG.hasMapping)}
    if maps != {LD.mapping_iri(m["mapping_id"]) for m in d["attack"]}: mism.append((c, "mappings"))
    techs = {str(t) for m in g1.objects(v, TG.hasMapping) for t in g1.objects(m, TG.technique)}
    if techs != {LD.technique_iri(m["technique_id"]) for m in d["attack"]}: mism.append((c, "techniques"))
    cw = {str(x) for x in g1.objects(v, TG.weakness)}
    if cw != {LD.cwe_iri(w["cwe_id"]) for w in d["kev_cwes"]}: mism.append((c, "cwe"))
check("jsonld", "facts read back from the JSON-LD graph (id, mappings, techniques, CWEs) == the JSON of the same CVE", not mism, mism[:3])

True

### 6.1 · Merge: the point of linked data
`/api/vulnerabilities/CVE-…` and `/api/techniques/T1190` are two **separate** documents. Both contain the mapping node `urn:threatgraph:pkev-mapping:<n>`:
the first adds the CVE side + technique + tactics, the second adds the CVE link. Loaded into one graph they merge into one node — nothing to join by hand.

In [11]:
cve = "CVE-2024-34102"; mid = next(m["mapping_id"] for m in docs[cve]["attack"] if m["technique_id"] == "T1190")
g_v = Graph().parse(data=json.dumps(api.vulnerability(cve, jsonld=True)), format="json-ld")
g_t = Graph().parse(data=json.dumps(api.technique("T1190", jsonld=True)), format="json-ld")
g_u = g_v + g_t; m_ = URIRef(LD.mapping_iri(mid))
props = lambda g: {str(p).split("#")[-1] for p, o in g.predicate_objects(m_)}
print("mapping node in vulnerability doc :", sorted(props(g_v)))
print("mapping node in technique doc     :", sorted(props(g_t)))
print("mapping node in the merged graph  :", sorted(props(g_u)))
check("merge", "same @id in two documents -> one node in the merged graph, with the properties of BOTH (vulnerability link from the technique doc; technique + capabilityGroup from the CVE doc)",
      {"vulnerability", "technique", "mappingType", "capabilityGroup"} <= props(g_u) and "vulnerability" not in props(g_v) and "capabilityGroup" not in props(g_t))
check("merge", "merged graph is smaller than the sum of the two (shared nodes are not duplicated)", len(g_u) < len(g_v) + len(g_t), (len(g_v), len(g_t), len(g_u)))

mapping node in vulnerability doc : ['capabilityGroup', 'comments', 'evidenceUrl', 'mappingType', 'technique', 'type', 'wasDerivedFrom']
mapping node in technique doc     : ['mappingType', 'technique', 'type', 'vulnerability', 'wasDerivedFrom']
mapping node in the merged graph  : ['capabilityGroup', 'comments', 'evidenceUrl', 'mappingType', 'technique', 'type', 'vulnerability', 'wasDerivedFrom']


True

## 7 · Dataset description (DCAT + PROV) — provenance for FAIR F2 / R1.2

In [12]:
ds = api.dataset(); dsn = {k: v for k, v in ds.items() if k != "@context"}
print(json.dumps({k: (v if k not in ("tables", "distribution") else f"[{len(v)} items]") for k, v in dsn.items() if k != "wasDerivedFrom"}, indent=1)[:900])
derived = {d["@id"]: d for d in dsn["wasDerivedFrom"]}
exp_iri = LD.prov_iris(SRC)
okprov = set(derived) == set(exp_iri.values()) and all(
    derived[exp_iri[k]]["version"] == s["pinned_ref"]
    and derived[exp_iri[k]].get("sha256") == next((f["sha256"] for f in s["files"] if f["role"] == "data"), None)
    for k, s in SRC["sources"].items())
check("dataset", "4 upstream sources with commit and data-file SHA-256 == config/sources.yaml (NVD has 1,391 files: no single hash, by design)", okprov, list(derived))
check("dataset", "dataset carries the raw-manifest and cleaning-code SHA-256 of the Phase-2 manifest (covers all 1,391 NVD files)",
      dsn["rawManifestSha256"] == clean_manifest["raw_manifest_sha256"] and dsn["cleaningCodeSha256"] == clean_manifest["cleaning_code_sha256"])
tabs = {t["name"]: t for t in dsn["tables"]}
check("dataset", "9 clean tables listed with SHA-256 == Phase-2 manifest and rowCount == rows in the database",
      set(tabs) == set(clean_manifest["tables"]) and all(tabs[t]["sha256"] == m["sha256"] and tabs[t]["rowCount"] == m["rows"] for t, m in clean_manifest["tables"].items()))
check("dataset", "dataset document is valid JSON-LD (no undefined terms) and parses as a dcat:Dataset with 4 prov:wasDerivedFrom",
      not LD.undefined_terms(dsn) and len(list(Graph().parse(data=json.dumps(ds), format="json-ld").subject_objects(URIRef("http://www.w3.org/ns/prov#wasDerivedFrom")))) == 4)
dead = []
for dist in dsn["distribution"]:
    for mt in dist["mediaType"]:
        r = requests.get(dist["accessURL"], headers={"Accept": mt}, params={"page_size": 1})
        if r.status_code != 200 or not r.headers["Content-Type"].startswith(mt): dead.append((dist["accessURL"], mt, r.status_code))
check("dataset", "every dcat:accessURL answers 200 in every advertised media type (the metadata does not lie)", not dead, dead)

{
 "@id": "http://127.0.0.1:45381/api/dataset/",
 "@type": "Dataset",
 "title": "ThreatGraph-LD: CISA KEV x NVD x MITRE ATT&CK x CTID PKEV",
 "description": "Open cyber-threat datasets integrated through the CVE identifier (KEV, NVD) and the ATT&CK technique identifier (PKEV, ATT&CK). Snapshot 2025-07-28.",
 "version": "2025-07-28",
 "modified": "2026-10-21T12:09:35Z",
 "rawManifestSha256": "f37bf97164198890f97e4135f5e1a128237e74b1e09afbc6c713db8a1f6e82e4",
 "cleaningCodeSha256": "2fb1c1f149e36af58a8937c246dff9555fa47c4c62633fedc6cea567e7f17cfa",
 "distribution": "[4 items]",
 "tables": "[9 items]"
}


True

## 8 · Efficiency: no N+1 queries
`select_related` / `prefetch_related` / `annotate` must make the **number of SQL queries independent of the page size**. Counted in-process with Django's test client.

In [13]:
setup_test_environment(); dj = DjangoClient()
def nq(url):
    with CaptureQueriesContext(connection) as c: r = dj.get(url)
    assert r.status_code == 200, (url, r.status_code); return len(c)
rows = [{"request": u, "SQL queries": nq(u)} for u in [
  "/api/vulnerabilities/?page_size=5", "/api/vulnerabilities/?page_size=200",
  "/api/vulnerabilities/CVE-2024-34102/", "/api/vulnerabilities/" + max(docs, key=lambda c: len(docs[c]["attack"])) + "/",
  "/api/techniques/?page_size=5", "/api/techniques/?page_size=200", "/api/techniques/T1190/", "/api/techniques/T1059.001/",
  "/api/vulnerabilities/?tactic=initial-access&page_size=200", "/api/cwes/?page_size=200"]]
qdf = pd.DataFrame(rows); display(qdf)
g = qdf.set_index("request")["SQL queries"]
check("perf", "list queries do not grow with page size (5 vs 200 rows): vulnerabilities and techniques",
      g["/api/vulnerabilities/?page_size=5"] == g["/api/vulnerabilities/?page_size=200"] and g["/api/techniques/?page_size=5"] == g["/api/techniques/?page_size=200"])
check("perf", "detail queries are constant (<= 12) whatever the number of mappings / references", qdf.iloc[[2, 3, 6, 7]]["SQL queries"].max() <= 12, qdf.iloc[[2, 3, 6, 7]]["SQL queries"].tolist())
# latency over real HTTP (local dev server, one client: a sanity figure, NOT a benchmark)
def lat(path, n=30, **kw):
    t = []
    for _ in range(n):
        s = time.perf_counter(); api.request(path, **kw).raise_for_status(); t.append((time.perf_counter() - s) * 1000)
    return round(statistics.median(t), 1), round(max(t), 1)
lt = pd.DataFrame([{"request": n, "median ms": lat(p, **kw)[0], "max ms": lat(p, **kw)[1]} for n, p, kw in [
  ("list page (50)", "/api/vulnerabilities/", {}), ("list filtered (technique=T1190)", "/api/vulnerabilities/?technique=T1190", {}),
  ("detail JSON", "/api/vulnerabilities/CVE-2024-34102/", {}), ("detail JSON-LD", "/api/vulnerabilities/CVE-2024-34102/", {"accept": JSONLD}),
  ("technique T1190 (157 mappings)", "/api/techniques/T1190/", {})]]); display(lt)
check("perf", "latency sanity: every request type has a median below 1 s on the local dev server", (lt["median ms"] < 1000).all(), lt["median ms"].tolist())

,request,SQL queries
0,/api/vulnerabilities/?page_size=5,2
1,/api/vulnerabilities/?page_size=200,2
2,/api/vulnerabilities/CVE-2024-34102/,7
3,/api/vulnerabilities/CVE-2021-40539/,7
4,/api/techniques/?page_size=5,3
5,/api/techniques/?page_size=200,3
6,/api/techniques/T1190/,4
7,/api/techniques/T1059.001/,4
8,/api/vulnerabilities/?tactic=initial-access&page_size=200,2
9,/api/cwes/?page_size=200,2


,request,median ms,max ms
0,list page (50),20.3,44.4
1,list filtered (technique=T1190),10.4,11.5
2,detail JSON,7.4,8.4
3,detail JSON-LD,7.7,10.1
4,technique T1190 (157 mappings),6.2,28.7


True

## 9 · Client demo
What a user of the library writes. The same calls power the checks above.

In [14]:
top = pd.DataFrame(api.get("/api/techniques/", {"ordering": "-n_vulnerabilities", "page_size": 10})["results"])
display(top[["technique_id", "name", "tactics", "n_vulnerabilities"]])
crit = pd.DataFrame(api.iter_vulnerabilities(technique="T1190", severity="CRITICAL"))
print(len(crit), "CRITICAL KEV CVEs mapped to T1190 (Exploit Public-Facing Application)"); display(crit[["cve_id", "vendor_project", "product", "cvss_v31_score"]].head(5))
try: api.vulnerability("CVE-1999-0001")
except ApiError as e: print("ApiError:", e.status, "-", str(e)[:90])

,technique_id,name,tactics,n_vulnerabilities
0,T1059,Command and Scripting Interpreter,[execution],170
1,T1190,Exploit Public-Facing Application,[initial-access],157
2,T1068,Exploitation for Privilege Escalation,[privilege-escalation],69
3,T1005,Data from Local System,[collection],46
4,T1078,Valid Accounts,"[initial-access, persistence, privilege-escalation, defense-evasion]",46
5,T1203,Exploitation for Client Execution,[execution],43
6,T1105,Ingress Tool Transfer,[command-and-control],35
7,T1204.002,Malicious File,[execution],33
8,T1505.003,Web Shell,[persistence],26
9,T1133,External Remote Services,"[initial-access, persistence]",25


98 CRITICAL KEV CVEs mapped to T1190 (Exploit Public-Facing Application)


,cve_id,vendor_project,product,cvss_v31_score
0,CVE-2010-2861,Adobe,ColdFusion,9.8
1,CVE-2013-0625,Adobe,ColdFusion,9.8
2,CVE-2013-0632,Adobe,ColdFusion,9.8
3,CVE-2014-6271,GNU,Bourne-Again Shell (Bash),9.8
4,CVE-2014-7169,GNU,Bourne-Again Shell (Bash),9.8


ApiError: 404 - HTTP 404 for http://127.0.0.1:45381/api/vulnerabilities/CVE-1999-0001/: {'detail': 'No Vul


## 10 · Negative tests — do the checks really fail on broken data?
A check that cannot fail proves nothing. Each test below damages a copy and expects the matching check to **raise a difference**.

In [15]:
neg = []
def negative(name, detected, detail=""):
    neg.append({"damage": name, "detected": bool(detected), "detail": str(detail)[:120]}); check("negative", name, detected, detail)
d = copy.deepcopy(docs["CVE-2024-34102"]); d["nvd"]["cvss_v31"]["score"] += 1
negative("CVSS score changed by +1 -> compare_vulnerability reports it", "cvss_v31" in compare_vulnerability(d, "CVE-2024-34102"))
d = copy.deepcopy(docs["CVE-2024-34102"]); d["attack"].pop()
negative("one PKEV mapping removed -> reported", "mappings" in compare_vulnerability(d, "CVE-2024-34102"))
d = copy.deepcopy(docs["CVE-2024-34102"]); d["ransomware_use"] = "known"
negative("ransomware flag flipped -> reported", "ransomware_use" in compare_vulnerability(d, "CVE-2024-34102"))
d = copy.deepcopy(docs["CVE-2024-34102"]); d["attack"][0]["references"].append({"reference": "https://fake.example", "is_http_url": True})
negative("invented evidence URL -> reported", any(x.startswith("evidence") for x in compare_vulnerability(d, "CVE-2024-34102")))
t = copy.deepcopy(tdocs["T1190"]); t["mappings"].pop()
negative("technique loses a CVE -> compare_technique reports it", "mappings" in compare_technique(t, "T1190"))
# the silent-drop pitfall
broken_ctx = {k: v for k, v in LD.CONTEXT.items() if k != "cveId"}
doc = {k: v for k, v in ld["CVE-2024-34102"].items() if k != "@context"}
negative("context without 'cveId': undefined_terms flags it", "$.cveId" in LD.undefined_terms(doc, broken_ctx))
n_ok = len(Graph().parse(data=json.dumps({"@context": LD.CONTEXT, **doc}), format="json-ld"))
n_bad = len(Graph().parse(data=json.dumps({"@context": broken_ctx, **doc}), format="json-ld"))
negative("the same broken context makes a JSON-LD processor DROP the triple SILENTLY (no error)", n_bad < n_ok, f"{n_ok} -> {n_bad} triples")
ctx2 = {**LD.CONTEXT, "ident": "dcterms:identifier"}
negative("two terms for one IRI ('cveId' and 'ident') -> terms_sharing_an_iri flags it", bool(LD.terms_sharing_an_iri(ctx2)))
negative("extra key 'foo' in a document -> undefined_terms flags it", "$.foo" in LD.undefined_terms({**doc, "foo": 1}))
negative("filter check can fail: pandas set of 'technique=T1190' is not the set of 'T1059'", {v["cve_id"] for v in api.iter_vulnerabilities(technique="T1059")} != S(pm[pm.technique_id == "T1190"]))
r = api.request("/api/vulnerabilities/CVE-2024-34102/", method="POST")
negative("write attempt (POST) on the read-only API -> 405 and data unchanged", r.status_code == 405 and M.Vulnerability.objects.count() == 1391)
display(pd.DataFrame(neg))

,damage,detected,detail
0,CVSS score changed by +1 -> compare_vulnerability reports it,True,
1,one PKEV mapping removed -> reported,True,
2,ransomware flag flipped -> reported,True,
3,invented evidence URL -> reported,True,
4,technique loses a CVE -> compare_technique reports it,True,
5,context without 'cveId': undefined_terms flags it,True,
6,the same broken context makes a JSON-LD processor DROP the triple ...,True,86 -> 85 triples
7,two terms for one IRI ('cveId' and 'ident') -> terms_sharing_an_ir...,True,
8,extra key 'foo' in a document -> undefined_terms flags it,True,
9,filter check can fail: pandas set of 'technique=T1190' is not the ...,True,


## 11 · Phase gate

In [16]:
rep = pd.DataFrame(checks)
summary = rep.groupby("family").ok.agg(passed="sum", total="count"); display(summary)
failed = rep[~rep.ok]; print("FAILED:", failed.to_dict("records") if len(failed) else "none")
server.terminate(); server.wait(timeout=10)
assert failed.empty, "Phase 4 gate failed (server log: %s)" % log.name
out = ROOT / "docs" / "phase4_validation_report.json"
out.write_text(json.dumps({"phase": 4, "django": django.get_version(), "all_passed": True, "summary": summary.reset_index().to_dict("records"),
                           "checks": checks, "clean_manifest_sha256": sha256_bytes((CLEAN / "manifest.json").read_bytes())}, indent=2) + "\n")
print("PHASE 4 GATE: PASSED ->", len(checks), "checks | report:", out.relative_to(ROOT))

,passed,total
family,,
content,5,5
dataset,5,5
filters,4,4
http,2,2
jsonld,12,12
merge,2,2
negative,11,11
paging,7,7
perf,3,3


FAILED: none
PHASE 4 GATE: PASSED -> 51 checks | report: docs/phase4_validation_report.json


## 12 · What this does **not** prove
* The API is exactly as correct as the clean layer: it inherits every limit of Phase 2 (consistent ≠ true; PKEV covers only 419 of 1,391 CVEs; NVD comes from an unofficial mirror).
* The vocabulary namespace `https://example.org/threatgraph/vocab#` is a **placeholder**; the `tg:` terms are not published or dereferenceable. A real deployment needs a persistent namespace and an ontology document [UNCERTAIN].
* Mapping IRIs `urn:threatgraph:pkev-mapping:<n>` are positional: stable only for the pinned PKEV commit.
* No authentication, rate limiting, caching or HTTPS: this is a read-only demo server (`runserver` is not for production). No OpenAPI schema yet.
* SQLite and Python 3.13 only; latency figures are single-client sanity values, not benchmarks.
* JSON-LD was validated with **rdflib** and **PyLD**; no SHACL shapes (a formal schema for the graph) were written.

**Next (Phase 5, concept only):** snapshots and diff (what changes between two dated versions of the four sources), then results/plots.